# Sentiment Classifier — WITHOUT TF-IDF (beginner version)

This notebook is the **simple** companion to `sentiment_notebook.ipynb`.
It does the same task — classify a sentence as **Positive / Negative / Neutral** —
but with the simplest possible recipe:

> **just count words  →  Naive Bayes  →  answer**

No TF-IDF, no `src/` pipeline, no vocabulary/vectorizer/stemmer classes.
Everything is written inline with plain Python so you can read it top to bottom.
It mirrors the standalone script `main_no_tfidf.py`.

**Setup:** launch Jupyter from the **project root** (the folder with `data/`) and *Run All*.

## 1. Load the data

Each line of the files looks like `some sentence<TAB>1`, where `1` = positive and `0` = negative.
We train on **imdb + amazon** and test on **yelp** (an unseen set).

In [1]:
def load_file(path):
    sentences, labels = [], []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            sentence, label = line.split('\t')
            sentences.append(sentence)
            labels.append(int(label))
    return sentences, labels

s1, l1 = load_file('data/imdb_labelled.txt')
s2, l2 = load_file('data/amazon_cells_labelled.txt')
train_sentences = s1 + s2
train_labels    = l1 + l2
test_sentences, test_labels = load_file('data/yelp_labelled.txt')

print(f'Training on {len(train_sentences)} sentences (imdb + amazon)')
print(f'Testing  on {len(test_sentences)} sentences (yelp)')
train_sentences[0], train_labels[0]

Training on 2000 sentences (imdb + amazon)
Testing  on 1000 sentences (yelp)


('A very, very, very slow-moving, aimless movie about a distressed, drifting young man.  ',
 0)

## 2. Turn a sentence into simple words

Lowercase, keep only letters, then split on spaces. That's the whole preprocessing.

In [2]:
def to_words(sentence):
    clean = ''
    for ch in sentence.lower():
        clean += ch if ch.isalpha() else ' '
    return clean.split()

to_words('Wow!! Loved this PLACE... 5 stars.')

['wow', 'loved', 'this', 'place', 'stars']

## 3. Train Naive Bayes — which is *just counting*

For each class we count how often every word appears, plus how many sentences are in each class.
That's literally all 'training' is here.

In [ ]:
def train(sentences, labels):
    word_counts = {0: {}, 1: {}}   # word -> count, per class
    total_words = {0: 0, 1: 0}     # total words seen in each class
    doc_counts  = {0: 0, 1: 0}     # number of sentences in each class
    vocabulary  = set()            # every word we have ever seen

    for sentence, label in zip(sentences, labels):
        doc_counts[label] += 1
        for word in to_words(sentence):
            vocabulary.add(word)
            word_counts[label][word] = word_counts[label].get(word, 0) + 1
            total_words[label] += 1

    return word_counts, total_words, doc_counts, vocabulary

model = train(train_sentences, train_labels)
word_counts, total_words, doc_counts, vocabulary = model
print(f'Vocabulary size : {len(vocabulary)} words')
print(f'Positive sentences: {doc_counts[1]}, Negative sentences: {doc_counts[0]}')

## 4. Classify a new sentence

For each class we add up log-probabilities: the prior `P(class)` plus `P(word|class)` for every word,
using **add-one (Laplace) smoothing** so an unseen word never makes the whole thing zero.
If Positive and Negative come out too close, we answer **Neutral**.

In [4]:
import math

LABEL_NAMES = {0: 'Negative', 1: 'Positive', 2: 'Neutral'}

def classify(sentence, model, neutral_gap=0.20):
    word_counts, total_words, doc_counts, vocabulary = model
    vocab_size = len(vocabulary)
    total_docs = doc_counts[0] + doc_counts[1]

    scores = {}
    for label in (0, 1):
        score = math.log(doc_counts[label] / total_docs)   # prior
        for word in to_words(sentence):
            count = word_counts[label].get(word, 0)
            prob = (count + 1) / (total_words[label] + vocab_size)  # Laplace
            score += math.log(prob)
        scores[label] = score

    # log-scores -> normal probabilities that add up to 1
    biggest = max(scores.values())
    exp0 = math.exp(scores[0] - biggest)
    exp1 = math.exp(scores[1] - biggest)
    total = exp0 + exp1
    p_neg, p_pos = exp0 / total, exp1 / total

    if abs(p_pos - p_neg) < neutral_gap:
        label = 2
    else:
        label = 1 if p_pos > p_neg else 0
    return label, {0: p_neg, 1: p_pos}

def predict(text, width=24):
    label, probs = classify(text, model)
    print(f'\"{text}\"')
    print(f'  -> {LABEL_NAMES[label]}')
    for c in (1, 0):
        filled = int(round(probs[c] * width))
        bar = '#' * filled + '.' * (width - filled)
        print(f'  {LABEL_NAMES[c]:<9} {bar} {probs[c]*100:5.1f}%')
    print()

predict('I absolutely loved this movie, it was fantastic!')
predict('The food was terrible and the service was awful.')
predict('it is a phone')

"I absolutely loved this movie, it was fantastic!"
  -> Positive
  Positive  ########################  98.3%
  Negative  ........................   1.7%

"The food was terrible and the service was awful."
  -> Negative
  Positive  ........................   0.4%
  Negative  ########################  99.6%

"it is a phone"
  -> Neutral
  Positive  ##############..........  58.4%
  Negative  ##########..............  41.6%



### Try your ownEdit the string and re-run.


In [5]:
predict('the battery life is amazing but the screen is too dim')

"the battery life is amazing but the screen is too dim"
  -> Positive
  Positive  ###################.....  78.5%
  Negative  #####...................  21.5%



## 5. Evaluate on the Yelp test set

The test set has no neutral labels, so any Neutral answer is counted separately (not as correct).

In [6]:
correct = 0
neutral = 0
predictions = []
for sentence, true_label in zip(test_sentences, test_labels):
    label, _ = classify(sentence, model)
    predictions.append(label)
    if label == 2:
        neutral += 1
    elif label == true_label:
        correct += 1

total = len(test_sentences)
print(f'Correct (Positive/Negative) : {correct} / {total} = {correct/total*100:.1f}%')
print(f'Answered Neutral (not sure) : {neutral} / {total}')

Correct (Positive/Negative) : 694 / 1000 = 69.4%
Answered Neutral (not sure) : 147 / 1000


## 6. Confusion matrix (pure-Python heatmap)

Rows = true label, columns = predicted label. A denser block means a larger count.

In [7]:
labels = [0, 1, 2]
names = [LABEL_NAMES[l] for l in labels]
index = {l: i for i, l in enumerate(labels)}
matrix = [[0]*len(labels) for _ in labels]
for t, p in zip(test_labels, predictions):
    matrix[index[t]][index[p]] += 1

def show_heatmap(matrix, names, width=9):
    biggest = max((v for row in matrix for v in row), default=1) or 1
    shades = ' .:+#'
    label_w = max(len(n) for n in names)
    print('rows = true, cols = predicted   (denser = larger count)\n')
    print(' ' * (label_w+1) + ''.join(f'{n:^{width}}' for n in names))
    for n, row in zip(names, matrix):
        cells = ''
        for v in row:
            ch = shades[min(len(shades)-1, int(v / biggest * (len(shades)-1)))]
            cells += f'{ch*3} {v:>3} '.center(width)
        print(f'{n:>{label_w}} {cells}')

show_heatmap(matrix, names)

rows = true, cols = predicted   (denser = larger count)

         Negative Positive  Neutral 
Negative  ### 379       54       67 
Positive  ... 105  +++ 315       80 
 Neutral        0        0        0 


## 7. Top words the model leans on

The words that are most lopsided toward one class — i.e. far more common in positive than negative sentences (and vice-versa).

In [8]:
def top_words(label, other, n=12):
    vocab_size = len(vocabulary)
    scores = []
    for w in vocabulary:
        p_here  = (word_counts[label].get(w, 0) + 1) / (total_words[label] + vocab_size)
        p_other = (word_counts[other].get(w, 0) + 1) / (total_words[other] + vocab_size)
        scores.append((w, math.log(p_here) - math.log(p_other)))
    scores.sort(key=lambda x: x[1], reverse=True)
    return [w for w, _ in scores[:n]]

print('Positive:', ', '.join(top_words(1, 0)))
print('Negative:', ', '.join(top_words(0, 1)))

Positive: nice, excellent, wonderful, beautiful, love, great, works, liked, loved, played, enjoyed, awesome
Negative: worst, poor, bad, stupid, horrible, waste, sucks, crap, terrible, slow, junk, sucked


## 8. With vs without TF-IDF

```
sentiment_notebook.ipynb           (full)   ... -> counts -> TF-IDF -> Naive Bayes
sentiment_notebook_no_tfidf.ipynb  (this)   ... -> counts ----------> Naive Bayes
```

The only difference is the missing TF-IDF weighting step. Everything else is the same idea —
count words and ask Naive Bayes which class they belong to. Run both notebooks and compare the
accuracy numbers in section 5 to see what TF-IDF buys you.

_Fun fact: Multinomial Naive Bayes is mathematically meant for integer **counts**, so this
simpler version is actually the more 'textbook-correct' one._